# SisFall Data Processing and SVM Training

This notebook processes the SisFall dataset, trains a Support Vector Machine (SVM) model, and exports the model as C code for embedded systems like STM32.

### Step 1: Install Required Libraries
First, we need to install the necessary Python libraries: `pandas`, `numpy`, `scikit-learn`, and `micromlgen`.

In [1]:
!pip install pandas numpy scikit-learn micromlgen

  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


### Step 2: Prepare Your SisFall Data

Create a directory named `SisFall_Data` in the same location as this notebook (or specify the full path below). Copy all the `.txt` files from your SisFall dataset into this directory.

### Step 3: Run the AI Training Script

The following script will:
1.  **Read and Process Data**: It iterates through the `.txt` files in `SisFall_Data`.
    *   It filters out the last 3 columns (assuming they are redundant).
    *   It downsamples the data from 200Hz to 5Hz by averaging every 40 rows.
    *   It creates overlapping windows of 25 samples (5 seconds) with a step of 12 for robust training.
    *   Labels are assigned based on the filename prefix ('F' for fall, 'D' for daily activities).
2.  **Train SVM Model**: A linear Support Vector Machine (SVM) is trained on the processed data.
3.  **Export C Code**: The trained SVM model is then exported as a C header file (`model.h`) which can be used on STM32 microcontrollers.

In [6]:
"""Train a subject-independent linear SVM for the STM32 fall detector.

Expected directory layout::

    Fall Detection csv/
    |-- SA01/
    |   |-- D01_R01.csv
    |   `-- F01_R01.csv
    |-- SA02/
    `-- SE01/

Supported CSV layouts:
  * 6 numeric columns: accelerometer XYZ + gyroscope XYZ.
  * 9 SisFall columns: ADXL345 XYZ + MMA8451Q XYZ + ITG3200 XYZ.

Outputs:
  * svm_model_generated.txt: constants and C inference function.
  * svm_training_results.json: split, metrics and reproducibility metadata.
"""

from __future__ import annotations

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DATA_DIR = Path("./Fall Detection csv")
MODEL_OUTPUT = Path("svm_model_generated.txt")
RESULT_OUTPUT = Path("svm_training_results.json")

RANDOM_STATE = 42
TEST_SUBJECT_RATIO = 0.20
VALIDATION_SUBJECT_RATIO = 0.20
TARGET_RECALL = 0.95

ORIGINAL_SAMPLE_RATE_HZ = 200
AI_SAMPLE_RATE_HZ = 5
DOWNSAMPLE_FACTOR = ORIGINAL_SAMPLE_RATE_HZ // AI_SAMPLE_RATE_HZ  # 40
WINDOW_SIZE = 25                 # 25 samples / 5 Hz = 5 seconds
ADL_WINDOW_STEP = 12
FALL_WINDOW_OFFSETS = (-5, 0, 5)
FEATURE_COUNT = 9


def find_subject_id(root: str) -> str | None:
    """Return the nearest SAxx/SExx directory name in root."""
    for part in reversed(Path(root).parts):
        candidate = part.upper()
        if re.fullmatch(r"(?:SA|SE)\d{2}", candidate):
            return candidate
    return None


def find_activity_id(filename: str) -> str | None:
    """Extract Dxx/Fxx from names such as D01_R01.csv or SA01_F01_R01.csv."""
    stem = Path(filename).stem.upper()
    match = re.search(r"(?:^|_)([DF]\d{2})(?:_|$)", stem)
    return match.group(1) if match else None


def looks_like_saved_index(series: pd.Series) -> bool:
    """Detect a pandas index column (0..N-1 or 1..N)."""
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    if values.size < 3:
        return False
    return bool(
        np.allclose(values, np.arange(values.size))
        or np.allclose(values, np.arange(1, values.size + 1))
    )


def load_sensor_csv(filepath: Path) -> pd.DataFrame:
    """Load accelerometer and gyro channels without silently mixing sensors."""
    raw = pd.read_csv(filepath, header=None, sep=",", engine="python")

    # SisFall text converted to CSV can retain a trailing semicolon.
    raw = raw.replace({r";\s*$": ""}, regex=True)
    numeric = raw.apply(pd.to_numeric, errors="coerce")
    numeric = numeric.dropna(axis=0, how="all").dropna(axis=1, how="all")

    # Remove an index accidentally saved by pandas.
    if numeric.shape[1] in (7, 10) and looks_like_saved_index(numeric.iloc[:, 0]):
        numeric = numeric.iloc[:, 1:]

    column_count = numeric.shape[1]
    if column_count == 6:
        # Converted layout: acceleration XYZ + gyroscope XYZ.
        selected = numeric.iloc[:, [0, 1, 2, 3, 4, 5]].copy()
    elif column_count == 9:
        # Original SisFall layout:
        # ADXL345 XYZ, MMA8451Q XYZ, ITG3200 XYZ.
        selected = numeric.iloc[:, [0, 1, 2, 6, 7, 8]].copy()
    else:
        raise ValueError(
            f"Expected 6 or 9 numeric columns, found {column_count}"
        )

    selected.columns = ["raw_x", "raw_y", "raw_z", "gx", "gy", "gz"]
    selected = selected.dropna().reset_index(drop=True)

    if selected.empty:
        raise ValueError("No valid numeric sensor rows")

    return selected


def preprocess_sensor_data(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same axis mapping and physical-unit scaling as the firmware."""
    processed = pd.DataFrame(index=df.index)

    # SisFall vertical Y axis -> physical device Ax axis.
    processed["ax"] = df["raw_y"] / 256.0
    processed["ay"] = df["raw_x"] / 256.0
    processed["az"] = df["raw_z"] / 256.0

    # ITG3200 scale used by SisFall: LSB/(degree/s).
    processed["gx"] = df["gx"] / 14.375
    processed["gy"] = df["gy"] / 14.375
    processed["gz"] = df["gz"] / 14.375

    # Mean pooling: 200 Hz -> 5 Hz, matching the firmware AI stream.
    bucket = np.arange(len(processed)) // DOWNSAMPLE_FACTOR
    return processed.groupby(bucket, sort=True).mean()


def build_windows(data: pd.DataFrame, label: int) -> list[pd.DataFrame]:
    """Create fall-centered windows or sliding ADL windows."""
    if len(data) < WINDOW_SIZE:
        return []

    windows: list[pd.DataFrame] = []

    if label == 1:
        acc_magnitude = np.sqrt(
            data["ax"].to_numpy() ** 2
            + data["ay"].to_numpy() ** 2
            + data["az"].to_numpy() ** 2
        )
        peak_position = int(np.argmax(acc_magnitude))
        starts: set[int] = set()

        for offset in FALL_WINDOW_OFFSETS:
            start = max(0, peak_position - 5 + offset)
            start = min(start, len(data) - WINDOW_SIZE)
            starts.add(start)

        for start in sorted(starts):
            windows.append(data.iloc[start:start + WINDOW_SIZE].copy())
    else:
        for start in range(0, len(data) - WINDOW_SIZE + 1, ADL_WINDOW_STEP):
            windows.append(data.iloc[start:start + WINDOW_SIZE].copy())

    return windows


def extract_features(window: pd.DataFrame) -> np.ndarray:
    """Extract the nine features implemented by the STM32 firmware."""
    ax = window["ax"].to_numpy(dtype=np.float64)
    ay = window["ay"].to_numpy(dtype=np.float64)
    az = window["az"].to_numpy(dtype=np.float64)
    gx = window["gx"].to_numpy(dtype=np.float64)
    gy = window["gy"].to_numpy(dtype=np.float64)
    gz = window["gz"].to_numpy(dtype=np.float64)

    acc_magnitude = np.sqrt(ax * ax + ay * ay + az * az)
    gyro_magnitude = np.sqrt(gx * gx + gy * gy + gz * gz)
    tilt = np.arctan2(np.sqrt(ay * ay + az * az), ax + 1.0e-6)
    tilt = np.degrees(tilt)

    # ddof=1 matches the sample standard deviation used in the firmware.
    features = np.array(
        [
            np.max(acc_magnitude),
            np.min(acc_magnitude),
            np.std(acc_magnitude, ddof=1),
            np.max(gyro_magnitude),
            np.std(gyro_magnitude, ddof=1),
            np.std(ax, ddof=1),
            np.std(ay, ddof=1),
            np.std(az, ddof=1),
            np.mean(tilt),
        ],
        dtype=np.float64,
    )

    if features.size != FEATURE_COUNT or not np.all(np.isfinite(features)):
        raise ValueError("Invalid feature vector")

    return features


def split_with_valid_classes(
    indices: np.ndarray,
    labels: np.ndarray,
    groups: np.ndarray,
    test_size: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Find a group-disjoint split in which both sides contain both classes."""
    for attempt in range(500):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed + attempt,
        )
        left_relative, right_relative = next(
            splitter.split(indices, labels[indices], groups=groups[indices])
        )
        left = indices[left_relative]
        right = indices[right_relative]

        if np.unique(labels[left]).size == 2 and np.unique(labels[right]).size == 2:
            return left, right

    raise RuntimeError(
        "Could not create a subject-disjoint split containing both classes. "
        "Check the available SA/SE subjects and labels."
    )


def choose_threshold(
    labels: np.ndarray,
    scores: np.ndarray,
    target_recall: float,
) -> tuple[float, float, float, float]:
    """Choose the highest-F1 validation threshold satisfying target recall."""
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    candidate_indices = np.flatnonzero(recall[:-1] >= target_recall)

    if candidate_indices.size == 0:
        raise RuntimeError(
            f"No validation threshold reaches recall {target_recall:.3f}"
        )

    candidate_precision = precision[candidate_indices]
    candidate_recall = recall[candidate_indices]
    candidate_f1 = (
        2.0
        * candidate_precision
        * candidate_recall
        / np.maximum(candidate_precision + candidate_recall, 1.0e-12)
    )
    best_local = int(np.argmax(candidate_f1))
    best_index = int(candidate_indices[best_local])

    return (
        float(thresholds[best_index]),
        float(precision[best_index]),
        float(recall[best_index]),
        float(candidate_f1[best_local]),
    )


def format_c_array(name: str, values: np.ndarray) -> str:
    joined = ", ".join(f"{float(value):.9f}f" for value in values)
    return f"static const float {name}[{len(values)}] = {{ {joined} }};"


def export_c_model(
    scaler: StandardScaler,
    classifier: SVC,
    threshold: float,
) -> str:
    scale = np.asarray(scaler.scale_, dtype=np.float64)
    if scale.size != FEATURE_COUNT or np.any(scale <= 0) or not np.all(np.isfinite(scale)):
        raise ValueError("Invalid scaler scale values")

    inverse_scale = 1.0 / scale
    coefficients = np.asarray(classifier.coef_[0], dtype=np.float64)
    bias = float(classifier.intercept_[0])

    lines = [
        "// Generated by train_sisfall_svm_subject_independent.py",
        f"#define AI_FEATURE_COUNT {FEATURE_COUNT}U",
        f"static const float SVM_BIAS = {bias:.9f}f;",
        f"static const float SVM_THRESHOLD = {threshold:.9f}f;",
        format_c_array("SVM_SCALE_MEAN", np.asarray(scaler.mean_)),
        format_c_array("SVM_INV_SCALE_STD", inverse_scale),
        format_c_array("SVM_COEF", coefficients),
        "",
        "uint8_t AI_Predict_Fall(const float *features) {",
        "    float score = SVM_BIAS;",
        "    for (uint8_t i = 0U; i < AI_FEATURE_COUNT; i++) {",
        "        float scaled_feature =",
        "            (features[i] - SVM_SCALE_MEAN[i]) * SVM_INV_SCALE_STD[i];",
        "        score += scaled_feature * SVM_COEF[i];",
        "    }",
        "    return (score > SVM_THRESHOLD) ? 1U : 0U;",
        "}",
        "",
    ]
    output = "\n".join(lines)
    MODEL_OUTPUT.write_text(output, encoding="utf-8")
    return output


def main() -> None:
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(
            f"Dataset directory not found: {DATA_DIR.resolve()}"
        )

    feature_rows: list[np.ndarray] = []
    labels: list[int] = []
    subject_groups: list[str] = []
    files_processed = 0
    files_skipped = 0

    print(f"Reading dataset: {DATA_DIR.resolve()}")

    for root, _, files in os.walk(DATA_DIR):
        subject_id = find_subject_id(root)
        if subject_id is None:
            continue

        for filename in sorted(files):
            if not filename.lower().endswith(".csv"):
                continue

            activity_id = find_activity_id(filename)
            if activity_id is None:
                print(f"[SKIP] Cannot identify Dxx/Fxx: {Path(root) / filename}")
                files_skipped += 1
                continue

            label = 1 if activity_id.startswith("F") else 0
            filepath = Path(root) / filename

            try:
                raw_data = load_sensor_csv(filepath)
                processed_data = preprocess_sensor_data(raw_data)
                windows = build_windows(processed_data, label)

                for window in windows:
                    feature_rows.append(extract_features(window))
                    labels.append(label)
                    subject_groups.append(subject_id)

                files_processed += 1
            except Exception as error:
                print(f"[SKIP] {filepath}: {error}")
                files_skipped += 1

    if not feature_rows:
        raise RuntimeError("No training windows were created")

    X = np.vstack(feature_rows)
    y = np.asarray(labels, dtype=np.uint8)
    groups = np.asarray(subject_groups)

    if not (len(X) == len(y) == len(groups)):
        raise RuntimeError("X, y and groups lengths are inconsistent")

    all_indices = np.arange(len(X))
    train_validation_indices, test_indices = split_with_valid_classes(
        all_indices,
        y,
        groups,
        TEST_SUBJECT_RATIO,
        RANDOM_STATE,
    )

    relative_validation_ratio = (
        VALIDATION_SUBJECT_RATIO / (1.0 - TEST_SUBJECT_RATIO)
    )
    train_indices, validation_indices = split_with_valid_classes(
        train_validation_indices,
        y,
        groups,
        relative_validation_ratio,
        RANDOM_STATE + 10_000,
    )

    train_subjects = sorted(set(groups[train_indices]))
    validation_subjects = sorted(set(groups[validation_indices]))
    test_subjects = sorted(set(groups[test_indices]))

    assert set(train_subjects).isdisjoint(validation_subjects)
    assert set(train_subjects).isdisjoint(test_subjects)
    assert set(validation_subjects).isdisjoint(test_subjects)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X[train_indices])
    X_validation_scaled = scaler.transform(X[validation_indices])
    X_test_scaled = scaler.transform(X[test_indices])

    classifier = SVC(
        kernel="linear",
        C=1.0,
        class_weight="balanced",
    )
    classifier.fit(X_train_scaled, y[train_indices])

    validation_scores = classifier.decision_function(X_validation_scaled)
    threshold, val_precision, val_recall, val_f1 = choose_threshold(
        y[validation_indices],
        validation_scores,
        TARGET_RECALL,
    )

    test_scores = classifier.decision_function(X_test_scaled)
    test_predictions = (test_scores > threshold).astype(np.uint8)

    report_dict = classification_report(
        y[test_indices],
        test_predictions,
        labels=[0, 1],
        target_names=["ADL", "Fall"],
        output_dict=True,
        zero_division=0,
    )
    report_text = classification_report(
        y[test_indices],
        test_predictions,
        labels=[0, 1],
        target_names=["ADL", "Fall"],
        zero_division=0,
    )
    matrix = confusion_matrix(y[test_indices], test_predictions, labels=[0, 1])
    tn, fp, fn, tp = (int(value) for value in matrix.ravel())
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    c_model = export_c_model(scaler, classifier, threshold)

    results = {
        "random_state": RANDOM_STATE,
        "data_directory": str(DATA_DIR),
        "files_processed": files_processed,
        "files_skipped": files_skipped,
        "total_windows": int(len(X)),
        "feature_count": FEATURE_COUNT,
        "target_recall": TARGET_RECALL,
        "validation_threshold": threshold,
        "validation_precision": val_precision,
        "validation_recall": val_recall,
        "validation_f1": val_f1,
        "train_subjects": train_subjects,
        "validation_subjects": validation_subjects,
        "test_subjects": test_subjects,
        "train_windows": int(len(train_indices)),
        "validation_windows": int(len(validation_indices)),
        "test_windows": int(len(test_indices)),
        "confusion_matrix": matrix.tolist(),
        "test_specificity": specificity,
        "classification_report": report_dict,
    }
    RESULT_OUTPUT.write_text(
        json.dumps(results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("\nSubject-independent split")
    print(f"  Train      ({len(train_subjects)}): {train_subjects}")
    print(f"  Validation ({len(validation_subjects)}): {validation_subjects}")
    print(f"  Test       ({len(test_subjects)}): {test_subjects}")
    print(f"\nValidation threshold: {threshold:.9f}")
    print(
        f"Validation precision={val_precision:.4f}, "
        f"recall={val_recall:.4f}, F1={val_f1:.4f}"
    )
    print("\nTEST REPORT")
    print(report_text)
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(matrix)
    print(f"Specificity: {specificity:.4f}")
    print(f"\nSaved: {MODEL_OUTPUT.resolve()}")
    print(f"Saved: {RESULT_OUTPUT.resolve()}")
    print("\nGenerated C model:\n")
    print(c_model)


if __name__ == "__main__":
    main()

Reading dataset: /content/Fall Detection csv

Subject-independent split
  Train      (22): [np.str_('SA02'), np.str_('SA03'), np.str_('SA04'), np.str_('SA06'), np.str_('SA08'), np.str_('SA10'), np.str_('SA11'), np.str_('SA12'), np.str_('SA13'), np.str_('SA15'), np.str_('SA16'), np.str_('SA17'), np.str_('SA18'), np.str_('SA19'), np.str_('SA23'), np.str_('SE02'), np.str_('SE03'), np.str_('SE06'), np.str_('SE07'), np.str_('SE10'), np.str_('SE12'), np.str_('SE13')]
  Validation (8): [np.str_('SA01'), np.str_('SA09'), np.str_('SA20'), np.str_('SA21'), np.str_('SA22'), np.str_('SE01'), np.str_('SE09'), np.str_('SE15')]
  Test       (8): [np.str_('SA05'), np.str_('SA07'), np.str_('SA14'), np.str_('SE04'), np.str_('SE05'), np.str_('SE08'), np.str_('SE11'), np.str_('SE14')]

Validation threshold: 0.298502437
Validation precision=0.9546, recall=0.9794, F1=0.9669

TEST REPORT
              precision    recall  f1-score   support

         ADL       1.00      0.99      0.99      2982
        Fall 

In [5]:
!unzip Falldetectioncsv.zip

Archive:  Falldetectioncsv.zip
   creating: Fall Detection csv/
  inflating: Fall Detection csv/.DS_Store  
   creating: Fall Detection csv/SA01/
  inflating: Fall Detection csv/SA01/D01_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D02_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D03_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D04_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R02.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R03.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R04.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R05.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R02.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R03.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R04.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R05.csv  
  inflating: Fall Detection csv/SA01/D07_SA01_R01.csv  
  inflating: F